# AirLLM vs Ollama vs HuggingFace — Benchmark Results Analysis

This notebook loads saved benchmark results, displays a summary table, and generates four comparison charts.

## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

matplotlib.use("Agg")  # headless backend — swap to 'inline' in interactive Jupyter

# Make the src package importable when running from notebooks/
sys.path.insert(0, str(Path('..') / 'src'))

from airllm_benchmark.visualization.chart_service import ChartService
from airllm_benchmark.visualization.notebook_data import NotebookDataPreparer

RESULTS_DIR = Path('..') / 'results'
ASSETS_DIR  = Path('..') / 'assets'

preparer = NotebookDataPreparer(results_dir=RESULTS_DIR)
charts   = ChartService(config={'assets_dir': str(ASSETS_DIR)})

print('Setup complete.')

## 2. Load Benchmark Results

In [ ]:
results = preparer.load_results()
print(f'Loaded {len(results)} result(s) from {RESULTS_DIR}')
for r in results:
    status = 'OK' if r.is_success else f'FAILED: {r.error}'
    print(f'  {r.method:<14} {r.model_id:<40} {status}')

## 3. Summary Table

In [ ]:
df = preparer.summary_dataframe()

if df.empty:
    print('No results found. Run airllm-benchmark first, then re-run this notebook.')
else:
    display_cols = ['method', 'model_id', 'latency_s', 'ram_peak_mb', 'vram_peak_mb',
                    'tokens_per_second', 'tokens_generated', 'cost_estimate', 'is_success']
    pd.set_option('display.max_colwidth', 50)
    display(df[display_cols].style.format({
        'latency_s':       '{:.2f}',
        'ram_peak_mb':     '{:.0f}',
        'vram_peak_mb':    '{:.0f}',
        'tokens_per_second': '{:.1f}',
    }))

## 4. Latency Bar Chart

In [ ]:
if results:
    path = charts.plot_latency_bar(results)
    print(f'Saved → {path}')
    img = plt.imread(str(path))
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No results to plot.')

## 5. Memory Comparison Chart (RAM vs VRAM)

In [ ]:
if results:
    path = charts.plot_memory_grouped_bar(results)
    print(f'Saved → {path}')
    img = plt.imread(str(path))
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No results to plot.')

## 6. Throughput Bar Chart (tokens/s)

In [ ]:
if results:
    path = charts.plot_throughput_bar(results)
    print(f'Saved → {path}')
    img = plt.imread(str(path))
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No results to plot.')

## 7. Trade-off Scatter Plot (Latency vs RAM)

In [ ]:
if results:
    path = charts.plot_trade_off_scatter(results)
    print(f'Saved → {path}')
    img = plt.imread(str(path))
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(img)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No results to plot.')

## 8. Cost Analysis and Conclusions

In [ ]:
print(preparer.hardware_profile_summary())
print()

if not df.empty:
    successful = df[df['is_success']]
    if not successful.empty:
        fastest = successful.loc[successful['latency_s'].idxmin(), 'method']
        most_efficient = successful.loc[successful['ram_peak_mb'].idxmin(), 'method']
        highest_tps = successful.loc[successful['tokens_per_second'].idxmax(), 'method']
        print(f'Fastest method       : {fastest}')
        print(f'Most RAM-efficient   : {most_efficient}')
        print(f'Highest throughput   : {highest_tps}')
        print()
        print('Cost estimates (energy at TDP):')
        for _, row in successful.iterrows():
            print(f'  {row["method"]:<14}: {row["cost_estimate"]}')
        print()
        print('Conclusion:')
        print('  AirLLM enables running models that exceed available VRAM via CPU layer paging,')
        print('  at the cost of significantly higher latency compared to Ollama or standard HF.')
        print('  Use Ollama for fast inference on small models; AirLLM for large models on low-VRAM hardware.')
    else:
        print('No successful results to analyse.')
else:
    print('Run airllm-benchmark to generate results, then re-run this notebook.')